# EDA — Healthcare Provider Fraud Detection

Análisis exploratorio del dataset `nudratabbas/healthcare-fraud-detection-dataset` ([Kaggle](https://www.kaggle.com/datasets/nudratabbas/healthcare-fraud-detection-dataset)).

**Requisitos:** ejecutar Jupyter con el entorno del proyecto (`uv sync` desde la raíz) y seleccionar el kernel de Python que apunta a `.venv` (no un Python del sistema sin dependencias).

Objetivo: estructura de datos, desbalance de clases y patrones útiles para features.

In [9]:
from __future__ import annotations

import os
import sys
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "healthcare_fraud").exists():
            return candidate
    return start

_root = _find_repo_root(Path.cwd().resolve())
# Raíz del repo (./data en .env sigue siendo <repo>/data, no notebooks/data)
os.environ["DATA_DIR"] = str((_root / "data").resolve())
_src = _root / "src"
if _src.is_dir():
    sp = str(_src)
    if sp not in sys.path:
        sys.path.insert(0, sp)

# Tras editar código en src/, re-ejecutar esta celda recarga el paquete sin reiniciar kernel
for _name in tuple(sys.modules):
    if _name == "healthcare_fraud" or _name.startswith("healthcare_fraud."):
        del sys.modules[_name]

import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from healthcare_fraud.data import clean_dataframe, load_dataset, validate_dataframe

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

## 1. Carga de datos

In [ ]:
# Carga tablas: CMS en data/raw/ (train_*…) o CSV único healthcare_fraud_detection.csv; si no hay CSV, descarga Kaggle
dfs_raw = load_dataset()
print("Tablas disponibles:", list(dfs_raw.keys()))

In [ ]:
# Validar y limpiar cada tabla
dfs = {}
for name, df in dfs_raw.items():
    validated = validate_dataframe(df, name)
    dfs[name] = clean_dataframe(validated, name)

summary = pd.DataFrame(
    [(k, *v.shape) for k, v in dfs.items()], columns=["tabla", "filas", "columnas"]
)
summary

## 2. Estructura de cada tabla

In [ ]:
for name, df in dfs.items():
    print(f"\n### {name} ###")
    display(df.dtypes.to_frame("dtype").T)
    display(df.head(2))

## 3. Desbalance de clases (variable objetivo)

In [ ]:
labels = dfs.get("labels_train")
claims_flat = dfs.get("claims_flat")

if labels is not None and "PotentialFraud" in labels.columns:
    counts = labels["PotentialFraud"].value_counts()
    ratio = counts.get(1, 0) / len(labels) * 100
    print(f"Proveedores totales: {len(labels):,}")
    print(f"Fraude (1): {counts.get(1, 0):,}  |  No fraude (0): {counts.get(0, 0):,}")
    print(f"Ratio de fraude: {ratio:.2f}%")

    fig, ax = plt.subplots(figsize=(5, 4))
    counts.rename({1: "Fraude", 0: "No fraude"}).plot.bar(ax=ax, color=["#e74c3c", "#2ecc71"])
    ax.set_title("Distribución de la variable objetivo")
    ax.set_xlabel("")
    ax.set_ylabel("Proveedores")
    plt.tight_layout()
    plt.show()
elif claims_flat is not None and "Is_Fraud" in claims_flat.columns:
    counts = claims_flat["Is_Fraud"].value_counts()
    ratio = counts.get(1, 0) / len(claims_flat) * 100
    print(f"Reclamaciones totales: {len(claims_flat):,}")
    print(f"Fraude (1): {counts.get(1, 0):,}  |  No fraude (0): {counts.get(0, 0):,}")
    print(f"Ratio de fraude (por reclamación): {ratio:.2f}%")

    fig, ax = plt.subplots(figsize=(5, 4))
    counts.rename({1: "Fraude", 0: "No fraude"}).plot.bar(ax=ax, color=["#e74c3c", "#2ecc71"])
    ax.set_title("Distribución de Is_Fraud (por reclamación)")
    ax.set_xlabel("")
    ax.set_ylabel("Reclamaciones")
    plt.tight_layout()
    plt.show()

## 4. Distribuciones de montos de reclamación

In [ ]:
amount_specs = [
    ("inpatient", "InscClaimAmtReimbursed"),
    ("outpatient", "InscClaimAmtReimbursed"),
    ("claims_flat", "Claim_Amount"),
]

for table_name, amount_col in amount_specs:
    df = dfs.get(table_name)
    if df is None or amount_col not in df.columns:
        continue

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df[amount_col].dropna().plot.hist(bins=50, ax=axes[0], color="steelblue")
    axes[0].set_title(f"{table_name} — distribución {amount_col}")
    axes[0].set_xlabel("USD")

    df[amount_col].dropna().plot.box(ax=axes[1])
    axes[1].set_title(f"{table_name} — boxplot {amount_col}")
    plt.tight_layout()
    plt.show()

    print(df[amount_col].describe().to_string())

## 5. Análisis de nulos por columna

In [ ]:
for name, df in dfs.items():
    null_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
    null_pct = null_pct[null_pct > 0]
    if null_pct.empty:
        print(f"{name}: sin nulos")
        continue

    fig, ax = plt.subplots(figsize=(10, max(3, len(null_pct) * 0.4)))
    null_pct.plot.barh(ax=ax, color="salmon")
    ax.set_title(f"{name} — % nulos por columna")
    ax.set_xlabel("% nulos")
    plt.tight_layout()
    plt.show()

## 6. Heatmap de correlaciones (inpatient)

In [ ]:
df_corr = dfs.get("inpatient") or dfs.get("claims_flat")
corr_title = "inpatient" if dfs.get("inpatient") is not None else "claims_flat"
if df_corr is not None:
    numeric_cols = df_corr.select_dtypes(include="number").columns.tolist()
    if len(numeric_cols) > 1:
        corr = df_corr[numeric_cols].corr()
        fig, ax = plt.subplots(figsize=(min(14, len(numeric_cols)), min(12, len(numeric_cols))))
        sns.heatmap(corr, annot=False, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
        ax.set_title(f"Correlaciones — {corr_title}")
        plt.tight_layout()
        plt.show()

## 7. Distribución geográfica (beneficiary)

In [ ]:
bene = dfs.get("beneficiary")
geo_df = bene
geo_col = "State"

if geo_df is None and dfs.get("claims_flat") is not None:
    geo_df = dfs["claims_flat"]
    geo_col = "Patient_State"

if geo_df is not None and geo_col in geo_df.columns:
    top_states = geo_df[geo_col].value_counts().head(20)
    fig, ax = plt.subplots(figsize=(12, 5))
    top_states.plot.bar(ax=ax, color="cornflowerblue")
    ax.set_title("Top 20 estados")
    ax.set_xlabel("Código / estado")
    ax.set_ylabel("Registros")
    plt.tight_layout()
    plt.show()

## 8. Conclusiones (completar tras ejecutar)

- **Desbalance**: en datos CMS suele ser ~9–10 % a nivel proveedor; en el CSV consolidado (`claims_flat`) la etiqueta es por reclamación (`Is_Fraud`).
- **Tablas**: layout CMS (`beneficiary`, `inpatient`/`outpatient`, `labels_train`) o una sola tabla `claims_flat` para datasets tipo `healthcare_fraud_detection.csv`.
- **Calidad**: muchas columnas clínicas con altos nulos — la agregación a nivel proveedor reduce ruido.
- **Siguiente paso**: `02_baseline.ipynb` y features en `features/build.py` (orientados al layout CMS salvo que adaptes el pipeline al consolidado).